# RSNA Knee Abnormality Detection — Cleanlab-Filtered Training (Fold 0)

Trains KneeAnatomicalMoEClassifier (ConvNeXt-Small) with E11 report-supervision
and 2,634 high-confidence Cleanlab Silver label errors zero-weighted (Abs_Diff >= 0.50).


In [ ]:
"""Self-Contained Kaggle Training Script & Notebook Generator.

Trains KneeAbnormalityClassifier on RSNA Knee Abnormality Detection dataset.
Zero external imports or GitHub cloning required.
Outputs best_model_fold_0.pt to /kaggle/working/
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys
import re
import glob
import unicodedata
from typing import List, Dict, Tuple, Optional, Union

import numpy as np
import pandas as pd
import pydicom
import random
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

# ==============================================================================
# 1. CONFIGURATION & CONSTANTS
# ==============================================================================

TARGET_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture"
]

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

CONFIG = {
    "backbone": "convnext_small",
    "pretrained": True,
    "epochs": 6,          # E11: same as E04 so the only change is the report auxiliary loss
    "batch_size": 2,
    "accum_steps": 2,
    "num_slices": 16,
    "target_size": (256, 256),  # cache_v1 is baked at 256x256 (E09 lesson)
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "dropout": 0.25,
    "label_smoothing": 0.03,
    "num_workers": 2,
    "max_train_studies": None,  # Set e.g. 500 for quick test, or None for full dataset
    # Which validation AUC picks the best checkpoint: "silver" (held-out Jev labels, ~10x more
    # studies, less noisy) or "gold" (58 expert-labelled studies). Both are always reported.
    "select_metric": "silver",
    # Phase 2: Multilabel Stratified 5-Fold CV & Non-destructive Augmentation
    "n_splits": 5,
    "fold": 0,          # Fold index to train (0..4). E11 uses fold 0 to compare with E04
    "augment": True,     # Volume-consistent affine & photometric (affine +-7 deg, scale 0.95-1.05)
    "hflip": False,      # Task T5 (N2): Mirror-knee horizontal flip. Validated by T3 laterality audit (55% R / 45% L)
    # E11: Report supervision (IDEAS.md #6C). The image model also predicts a fingerprint of the
    # radiology report (char TF-IDF -> SVD, language-agnostic). Training-only; the report head is
    # not part of the model state_dict, so inference/submission code is unchanged.
    "report_aux": True,
    "report_dim": 64,
    "report_aux_weight": 0.5,
}




# ==============================================================================
# 2. PATH AUTODETECTION
# ==============================================================================

def get_train_paths() -> Tuple[str, str, str]:
    """Locate train.csv, train_series.csv, and train_series directory."""
    train_csv, series_csv, series_dir = None, None, None

    # 1. Search Kaggle input
    if os.path.exists("/kaggle/input"):
        print("Scanning /kaggle/input for competition data...")
        try:
            for entry in sorted(os.listdir("/kaggle/input")):
                print(f"  /kaggle/input/{entry}")
                ep = os.path.join("/kaggle/input", entry)
                if os.path.isdir(ep):
                    for sub in sorted(os.listdir(ep))[:10]:
                        print(f"    {entry}/{sub}")
        except OSError as e:
            print(f"  Error reading /kaggle/input: {e}")

        for root, dirs, files in os.walk("/kaggle/input"):
            # Prune DICOM/image folders to keep search instant
            dirs[:] = [d for d in dirs if not any(x in d.lower() for x in ["series", "image", "dicom", "dcm", "studies"])]
            if "train.csv" in files:
                train_csv = os.path.join(root, "train.csv")
                print(f"  --> Found train.csv at: {train_csv}")
                if "train_series.csv" in files:
                    series_csv = os.path.join(root, "train_series.csv")
                    print(f"  --> Found train_series.csv at: {series_csv}")
                for cand in ["train_series", "train_images", "train"]:
                    cand_p = os.path.join(root, cand)
                    if os.path.isdir(cand_p):
                        series_dir = cand_p
                        print(f"  --> Found train series dir at: {series_dir}")
                        break
                break

    # 2. Local fallback
    if train_csv is None or not os.path.exists(train_csv):
        local_root = os.path.abspath("data/raw")
        if os.path.exists(os.path.join(local_root, "train.csv")):
            train_csv = os.path.join(local_root, "train.csv")
            series_csv = os.path.join(local_root, "train_series.csv")
            series_dir = os.path.join(local_root, "sample_dicom")

    print(f"\nFinal Paths:")
    print(f"  Train CSV : {train_csv}")
    print(f"  Series CSV: {series_csv}")
    print(f"  Series Dir: {series_dir}")
    return train_csv, series_csv, series_dir


# ==============================================================================
# 3. MULTILINGUAL REGEX LABEL MINING (PURE PYTHON, 0 DEPENDENCIES)
# ==============================================================================

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text).lower()
    text = re.sub(r"[\r\n\t]+", " . ", text)
    text = re.sub(r"[;:]+", " . ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def strip_accents(text: str) -> str:
    text_nfkd = unicodedata.normalize("NFKD", text)
    return "".join(c for c in text_nfkd if not unicodedata.combining(c))

NEGATION_WORDS = [
    r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bfree of\b", r"\bnegative for\b",
    r"\babsence of\b", r"\bintact\b", r"\bunremarkable\b", r"\bnormal\b",
    r"\bpreserved\b", r"\buninjured\b", r"\bcontinuous\b", r"\bno evidence of\b",
    r"\bsin\b", r"\bausencia de\b", r"\bintacto\b", r"\bintacta\b", r"\bconservado\b",
    r"\bconservada\b", r"\brespetado\b", r"\brespetada\b", r"\bno se observa\b",
    r"\bno se aprecian\b", r"\bno hay\b", r"\bsin signos de\b", r"\bsans\b",
    r"\babsence de\b", r"\bconserve\b", r"\bconservee\b", r"\bpas de\b",
    r"\bgeen\b", r"\bzonder\b", r"\bnormaal\b", r"\bonopvallend\b",
    r"\byoktur\b", r"\bnormaldir\b", r"\byok\b", r"\bizlenmedi\b", r"\bsaptanmadi\b",
    r"\bχωρίς\b", r"\bδεν\b", r"\bφυσιολογικ", r"\bбез\b", r"\bняма\b", r"\bсъхранен"
]

HEDGE_WORDS = [
    r"cannot exclude", r"could not exclude", r"possible", r"equivocal",
    r"indeterminate", r"questionable", r"borderline", r"no descartable",
    r"dudoso", r"no excluible", r"pequena duda", r"non exclu", r"şüpheli",
    r"suspect", r"suspected", r"sospecha"
]

class MultilingualRegexLabeler:
    def __init__(self):
        self._compile_patterns()

    def _compile_patterns(self):
        acl_terms = r"(acl|lca|voorste kruisband|croise anterieur|anterior cruciate|ligamento cruzado anterior|anterior capraz|χιαστό|кръстни връзки)"
        tear_terms = r"(tear|ruptur|rotur|rupture|torn|disrupt|injury|avulsion|scheur|lesion|lacerat|ρήξη|разкъсване|фрактура)"
        intact_terms = r"(intact|normal|conservad|unremarkable|sin alteracion|sans particularite|behoud|integrite|normaldir|φυσιολογικ|съхранен)"

        self.acl_pos = re.compile(rf"\b{acl_terms}\b.{{0,60}}?{tear_terms}", re.IGNORECASE)
        self.acl_pos_rev = re.compile(rf"{tear_terms}.{{0,60}}?\b{acl_terms}\b", re.IGNORECASE)
        self.acl_neg = re.compile(rf"\b{acl_terms}\b.{{0,40}}?{intact_terms}", re.IGNORECASE)
        self.acl_neg_rev = re.compile(rf"(no|sin|sans|geen|yok|без|няма|{intact_terms}).{{0,40}}?\b{acl_terms}\b", re.IGNORECASE)

        mcl_terms = r"(mcl|lcm|medial collateral|ligamento colateral medial|mediale band|collat[eé]ral m[eé]dial|medial kollateral|πλάγιοι|коллатерал)"
        sprain_terms = r"(tear|ruptur|rotur|rupture|torn|disrupt|sprain|lesion|strain|esguince|entorse|injury|scheur|incelme|ödem|ρήξη)"
        self.mcl_pos = re.compile(rf"\b{mcl_terms}\b.{{0,60}}?{sprain_terms}", re.IGNORECASE)
        self.mcl_pos_rev = re.compile(rf"{sprain_terms}.{{0,60}}?\b{mcl_terms}\b", re.IGNORECASE)
        self.mcl_neg = re.compile(rf"\b{mcl_terms}\b.{{0,40}}?{intact_terms}", re.IGNORECASE)
        self.mcl_neg_rev = re.compile(rf"(no|sin|sans|geen|yok|без|няма|{intact_terms}).{{0,40}}?\b{mcl_terms}\b", re.IGNORECASE)

        mm_terms = r"(medial meniscus|menisco medial|menisco interno|binnenmeniscus|menisque medial|menisque interne|medial menisk|έσω διαμέρισμα|медиалния менискус|mm\b)"
        meniscus_tear = r"(tear|ruptur|rotur|rupture|torn|fissur|lesion|amputat|flap|scheur|cleavage|meniskopati|ρήξη|дегенерация|разкъсване|rotura)"
        self.mm_pos = re.compile(rf"\b{mm_terms}.{{0,60}}?{meniscus_tear}", re.IGNORECASE)
        self.mm_pos_rev = re.compile(rf"{meniscus_tear}.{{0,60}}?\b{mm_terms}", re.IGNORECASE)
        self.mm_neg = re.compile(rf"\b({mm_terms}|meniscos? (medial|interno|ambos|los dos)).{{0,40}}?{intact_terms}", re.IGNORECASE)
        self.mm_neg_rev = re.compile(rf"(no|sin|sans|geen|yok|без|няма|{intact_terms}).{{0,40}}?\b({mm_terms}|meniscos?)", re.IGNORECASE)

        lm_terms = r"(lateral meniscus|menisco lateral|menisco externo|buitenmeniscus|menisque lateral|menisque externe|lateral menisk|έξω διαμέρισμα|латералния менискус|lm\b)"
        self.lm_pos = re.compile(rf"\b{lm_terms}.{{0,60}}?{meniscus_tear}", re.IGNORECASE)
        self.lm_pos_rev = re.compile(rf"{meniscus_tear}.{{0,60}}?\b{lm_terms}", re.IGNORECASE)
        self.lm_neg = re.compile(rf"\b({lm_terms}|meniscos? (lateral|externo|ambos|los dos)|medial y lateral).{{0,40}}?{intact_terms}", re.IGNORECASE)
        self.lm_neg_rev = re.compile(rf"(no|sin|sans|geen|yok|без|няма|{intact_terms}).{{0,40}}?\b({lm_terms}|meniscos?|medial y lateral)", re.IGNORECASE)

        oa_signs = r"(osteoarthritis|oa|arthrosis|artrosis|gonartrosis|gonarthrose|chondromalacia|chondropathy|condropatia|chondrosis|cartilage loss|cartilage defect|ulcer|pinzamiento|thinning|wear|οστεοαρθρίτιδα|хрущял)"
        moa_terms = r"(medial|femorotibial medial|compartimento medial|compartiment medial|binnenste compartiment|condilo femoral medial|plateau tibial medial|έσω κνημιαίο)"
        self.moa_pos = re.compile(rf"\b{moa_terms}.{{0,60}}?{oa_signs}", re.IGNORECASE)
        self.moa_pos_rev = re.compile(rf"{oa_signs}.{{0,60}}?\b{moa_terms}", re.IGNORECASE)
        self.moa_neg = re.compile(rf"\b{moa_terms}.{{0,40}}?(normal|intact|preserved|conservad|no chondral|sin artrosis)", re.IGNORECASE)

        loa_terms = r"(lateral|femorotibial lateral|compartimento lateral|compartiment lateral|buitenste compartiment|condilo femoral lateral|plateau tibial lateral|έξω κνημιαίο)"
        self.loa_pos = re.compile(rf"\b{loa_terms}.{{0,60}}?{oa_signs}", re.IGNORECASE)
        self.loa_pos_rev = re.compile(rf"{oa_signs}.{{0,60}}?\b{loa_terms}", re.IGNORECASE)
        self.loa_neg = re.compile(rf"\b{loa_terms}.{{0,40}}?(normal|intact|preserved|conservad|no chondral|sin artrosis)", re.IGNORECASE)

        pfoa_terms = r"(patellofemoral|femoropatellar|femororrotuliana|femoropatelar|patellar cartilage|trochlear cartilage|rotuliana|rotula|patella|retropatellaire|patelofemoral|патела|trochlea)"
        self.pfoa_pos = re.compile(rf"\b{pfoa_terms}.{{0,60}}?{oa_signs}", re.IGNORECASE)
        self.pfoa_pos_rev = re.compile(rf"{oa_signs}.{{0,60}}?\b{pfoa_terms}", re.IGNORECASE)
        self.pfoa_neg = re.compile(rf"\b{pfoa_terms}.{{0,40}}?(normal|intact|preserved|conservad|no chondral|sin alteracion)", re.IGNORECASE)

        self.eff_pos = re.compile(r"\b(joint effusion|effusion|derrame|derrame articular|epanchement|hydrops|vochtuitstorting|fluid accumulation|fluid in the joint|liquid articular|sıvı artışı|υγρού|излив|bursitis|hidrartrosis)\b", re.IGNORECASE)
        self.eff_neg = re.compile(r"\b(no|not|sin|sans|geen|yok|без|няма|without|ausencia de).{0,30}?\b(joint effusion|effusion|derrame|epanchement|hydrops|fluid|sıvı|υγρού|излив)\b", re.IGNORECASE)

        self.syn_pos = re.compile(r"\b(synovitis|sinovitis|synoviale|synovial thickening|synovial hypertrophy|synovial enhancement|engrosamiento sinovial|hipertrofia sinovial|epaississement synovial|hoffitis|sinovit|συνοβιακ)\b", re.IGNORECASE)
        self.syn_neg = re.compile(r"\b(no|not|sin|sans|geen|yok|без|няма|without|ausencia de).{0,30}?\b(synovitis|sinovitis|synovial thickening|synovial hypertrophy|sinovit)\b", re.IGNORECASE)

        self.baker_pos = re.compile(r"\b(baker|bakers|popliteal cyst|quiste de baker|quiste popliteo|kyste poplite|kyste de baker|bakerse cyste|popliteale cyste|baker kisti|κύστη baker)\b", re.IGNORECASE)
        self.baker_neg = re.compile(r"\b(no|not|sin|sans|geen|yok|без|няма|without|ausencia de).{0,30}?\b(baker|bakers|popliteal cyst|quiste de baker|quiste popliteo|kyste poplite)\b", re.IGNORECASE)

        self.cont_pos = re.compile(r"\b(bone contusion|bone bruise|contusion osea|contusion osseuse|botcontusie|marrow edema|edema oseo|edema medular|bone marrow edema|oedeme osseux|osseous contusion|osteocondral contusion|kemik iliği ödem|οστεομυελικό οίδημα|костномозъчен едем|kissing contusion|contusion|contusiones)\b", re.IGNORECASE)
        self.cont_neg = re.compile(r"\b(no|not|sin|sans|geen|yok|без|няма|without|ausencia de).{0,30}?\b(bone contusion|bone bruise|contusion osea|marrow edema|edema oseo|edema medular|bone marrow edema)\b", re.IGNORECASE)

        self.frac_pos = re.compile(r"\b(fracture|fractura|fractur|fraktür|break|insufficiency fracture|subchondral fracture|trabecular fracture|fracture de fatigue|κατάγματα|фрактура)\b", re.IGNORECASE)
        self.frac_neg = re.compile(r"\b(no|not|sin|sans|geen|yok|без|няма|without|ausencia de).{0,30}?\b(fracture|fractura|fractur|fraktür|break|κατάγματα|фрактура)\b", re.IGNORECASE)

    def extract_single(self, text: str, pos_re, neg_re, pos_rev_re=None, neg_rev_re=None) -> Optional[float]:
        sentences = re.split(r"[.\n;•\->]+", text)
        has_pos, has_neg = False, False
        for s in sentences:
            s = s.strip()
            if not s:
                continue
            clean_s = strip_accents(s)
            if neg_re and neg_re.search(clean_s):
                has_neg = True
            elif neg_rev_re and neg_rev_re.search(clean_s):
                has_neg = True

            matched_pos = False
            if pos_re and pos_re.search(clean_s):
                matched_pos = True
            elif pos_rev_re and pos_rev_re.search(clean_s):
                matched_pos = True

            if matched_pos:
                hedge_p = r"(" + "|".join(HEDGE_WORDS) + r")"
                neg_p = r"(" + "|".join(NEGATION_WORDS) + r")"
                if re.search(hedge_p, clean_s, re.IGNORECASE) or re.search(neg_p, clean_s, re.IGNORECASE):
                    continue
                has_pos = True

        if has_pos:
            return 1.0
        elif has_neg:
            return 0.0
        return None

    def extract_from_report(self, report_text: str) -> Dict[str, Optional[float]]:
        norm = normalize_text(report_text)
        if not norm:
            return {t: None for t in TARGET_COLS}
        return {
            "ACL": self.extract_single(norm, self.acl_pos, self.acl_neg, self.acl_pos_rev, self.acl_neg_rev),
            "MCL": self.extract_single(norm, self.mcl_pos, self.mcl_neg, self.mcl_pos_rev, self.mcl_neg_rev),
            "Medial Meniscus": self.extract_single(norm, self.mm_pos, self.mm_neg, self.mm_pos_rev, self.mm_neg_rev),
            "Lateral Meniscus": self.extract_single(norm, self.lm_pos, self.lm_neg, self.lm_pos_rev, self.lm_neg_rev),
            "Medial OA": self.extract_single(norm, self.moa_pos, self.moa_neg, self.moa_pos_rev),
            "Lateral OA": self.extract_single(norm, self.loa_pos, self.loa_neg, self.loa_pos_rev),
            "PF OA": self.extract_single(norm, self.pfoa_pos, self.pfoa_neg, self.pfoa_pos_rev),
            "Effusion": self.extract_single(norm, self.eff_pos, self.eff_neg),
            "Synovitis": self.extract_single(norm, self.syn_pos, self.syn_neg),
            "Baker's": self.extract_single(norm, self.baker_pos, self.baker_neg),
            "Contusion": self.extract_single(norm, self.cont_pos, self.cont_neg),
            "Fracture": self.extract_single(norm, self.frac_pos, self.frac_neg),
        }


# === EMBEDDED_JEV_LABELS_START ===
EMBEDDED_JEV_LABELS_B64 = "..."
# === EMBEDDED_JEV_LABELS_END ===

def assemble_labels(train_csv_path: str) -> pd.DataFrame:
    """Build or load master train labels (Jev 0.878 AUC soft labels prioritized)."""
    # 1. Look for pre-assembled train_labels.csv
    candidates = [
        "data/processed/train_labels.csv",
        "train_labels.csv",
    ]
    if os.path.exists("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [d for d in dirs if not any(x in d.lower() for x in ["series", "image", "dicom", "dcm", "studies"])]
            if "train_labels.csv" in files:
                candidates.insert(0, os.path.join(root, "train_labels.csv"))

    for c in candidates:
        if os.path.isfile(c):
            print(f"Loading pre-assembled training labels from: {c}")
            out_df = pd.read_csv(c)
            if "ACL" in out_df.columns and "StudyInstanceUID" in out_df.columns:
                out_df["StudyInstanceUID"] = out_df["StudyInstanceUID"].astype(str)
                out_df["is_gold"] = out_df.get("label_source", "") == "gold"
                gold_count = out_df["is_gold"].sum()
                print(f"--> Successfully loaded {len(out_df)} studies | {gold_count} Gold | {(out_df.get('label_source')=='jev').sum()} Jev Silver")
                return out_df

    # 2. Check embedded Jev labels
    if "EMBEDDED_JEV_LABELS_B64" in globals() and EMBEDDED_JEV_LABELS_B64:
        try:
            import io, gzip, base64
            print("Unpacking embedded Jev System One labels (0.878 AUC)...")
            raw_bytes = gzip.decompress(base64.b64decode(EMBEDDED_JEV_LABELS_B64))
            out_df = pd.read_csv(io.BytesIO(raw_bytes))
            out_df["StudyInstanceUID"] = out_df["StudyInstanceUID"].astype(str)
            out_df["is_gold"] = out_df.get("label_source", "") == "gold"
            gold_count = out_df["is_gold"].sum()
            print(f"--> [CLEANLAB FILTERED] Successfully unpacked {len(out_df)} studies (2,634 Silver label errors zero-weighted)")
            print(f"--> Total zero-weighted target instances: {(out_df[[c for c in out_df.columns if c.endswith('_weight')]] == 0.0).sum().sum()}")
            print(f"--> Successfully unpacked {len(out_df)} studies | {gold_count} Gold | {(out_df.get('label_source')=='jev').sum()} Jev Silver")
            return out_df
        except Exception as e:
            print(f"Warning: Failed to unpack embedded labels: {e}")

    # 3. Fallback to multilingual regex extractor
    df = pd.read_csv(train_csv_path)

    labeler = MultilingualRegexLabeler()
    print(f"Fallback: Assembling regex labels for {len(df)} studies...")

    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Mining Labels"):
        uid = str(row["StudyInstanceUID"])
        out_row = {"StudyInstanceUID": uid}

        is_gold = not pd.isna(row["ACL"])
        reg_preds = labeler.extract_from_report(row.get("Report", "")) if not is_gold else {}

        gold_or_regex = False
        for t in TARGET_COLS:
            if is_gold and not pd.isna(row.get(t)):
                out_row[t] = float(row[t])
                out_row[f"{t}_weight"] = 1.0
                gold_or_regex = True
            elif t in reg_preds and reg_preds[t] is not None:
                out_row[t] = float(reg_preds[t])
                out_row[f"{t}_weight"] = 0.8
                gold_or_regex = True
            else:
                out_row[t] = 0.5
                out_row[f"{t}_weight"] = 0.2

        out_row["is_gold"] = is_gold
        out_row["label_source"] = "gold" if is_gold else ("regex" if gold_or_regex else "soft_unk")
        rows.append(out_row)

    out_df = pd.DataFrame(rows)
    gold_count = out_df["is_gold"].sum()
    print(f"--> Labels compiled: {len(out_df)} studies | {gold_count} Gold | {(out_df['label_source']=='regex').sum()} Regex")
    return out_df


# ==============================================================================
# 4. DICOM PREPROCESSING (ON-THE-FLY)
# ==============================================================================

def apply_windowing(pixels: np.ndarray, center=None, width=None, photometric: str = "MONOCHROME2") -> np.ndarray:
    img = pixels.astype(np.float32)
    if photometric == "MONOCHROME1":
        img = img.max() - img
    if isinstance(center, (list, pydicom.multival.MultiValue)):
        center = float(center[0])
    if isinstance(width, (list, pydicom.multival.MultiValue)):
        width = float(width[0])

    if center is not None and width is not None and width > 0:
        c, w = float(center), float(width)
        img_min = c - 0.5 - (w - 1) / 2.0
        img_max = c - 0.5 + (w - 1) / 2.0
        img = np.clip(img, img_min, img_max)
        img = (img - img_min) / (img_max - img_min) * 255.0
    else:
        vmin, vmax = np.percentile(img, 1.0), np.percentile(img, 99.0)
        if vmax > vmin:
            img = np.clip(img, vmin, vmax)
            img = (img - vmin) / (vmax - vmin) * 255.0
        else:
            img = np.zeros_like(img)
    return img.astype(np.uint8)

def crop_empty_borders(img: np.ndarray, threshold: int = 10, margin: int = 5) -> np.ndarray:
    mask = img > threshold
    if not np.any(mask):
        return img
    y_idx, x_idx = np.where(mask)
    h, w = img.shape[:2]
    ymin = max(0, y_idx.min() - margin)
    ymax = min(h, y_idx.max() + margin + 1)
    xmin = max(0, x_idx.min() - margin)
    xmax = min(w, x_idx.max() + margin + 1)
    return img[ymin:ymax, xmin:xmax]

def compute_slice_position(dcm: pydicom.Dataset) -> float:
    ori = getattr(dcm, "ImageOrientationPatient", [1, 0, 0, 0, 1, 0])
    pos = getattr(dcm, "ImagePositionPatient", [0, 0, 0])
    f = np.array(ori[:3], dtype=float)
    c = np.array(ori[3:], dtype=float)
    normal = np.cross(f, c)
    norm = np.linalg.norm(normal)
    normal = normal / norm if norm > 1e-6 else np.array([0.0, 0.0, 1.0])
    return float(np.dot(normal, np.array(pos, dtype=float)))

def load_and_preprocess_series(dicom_files: List[str], target_size=(256, 256), num_slices: int = 16) -> torch.Tensor:
    if not dicom_files:
        return torch.zeros((num_slices, 3, target_size[0], target_size[1]), dtype=torch.float32)

    slice_entries = []
    for fpath in dicom_files:
        try:
            dcm = pydicom.dcmread(fpath)
            coord = compute_slice_position(dcm)
            photometric = getattr(dcm, "PhotometricInterpretation", "MONOCHROME2")
            wc = getattr(dcm, "WindowCenter", None)
            ww = getattr(dcm, "WindowWidth", None)
            pixels = dcm.pixel_array.astype(np.float32)
            slope = float(getattr(dcm, "RescaleSlope", 1.0))
            intercept = float(getattr(dcm, "RescaleIntercept", 0.0))
            if slope != 1.0 or intercept != 0.0:
                pixels = pixels * slope + intercept
            img = apply_windowing(pixels, center=wc, width=ww, photometric=photometric)
            img = crop_empty_borders(img)
            img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
            slice_entries.append((coord, img))
        except Exception:
            continue

    if not slice_entries:
        return torch.zeros((num_slices, 3, target_size[0], target_size[1]), dtype=torch.float32)

    slice_entries.sort(key=lambda x: x[0])
    volume = np.stack([x[1] for x in slice_entries], axis=0)

    n_slices = len(volume)
    indices = np.linspace(0, n_slices - 1, num_slices).round().astype(int)
    slabs = []
    for idx in indices:
        z_prev = volume[max(0, idx - 1)]
        z_curr = volume[idx]
        z_next = volume[min(n_slices - 1, idx + 1)]
        slab = np.stack([z_prev, z_curr, z_next], axis=0).astype(np.float32) / 255.0
        slab = (slab - IMAGENET_MEAN) / IMAGENET_STD
        slabs.append(slab)

    return torch.tensor(np.stack(slabs, axis=0), dtype=torch.float32)


# ==============================================================================
# 5. PYTORCH DATASET
# ==============================================================================

def cached_plane_to_slabs(arr: np.ndarray) -> np.ndarray:
    """(K, H, W) uint8 cached slices -> (K, 3, H, W) float32 2.5D slabs, ImageNet-normalized.

    The neighbour channels are the adjacent *sampled* slices, not adjacent slices of the full
    series. The submission notebook has an identical copy; tests/test_preprocessing_parity.py
    checks they stay in sync.
    """
    K = arr.shape[0]
    slabs = []
    for idx in range(K):
        z_prev = arr[max(0, idx - 1)]
        z_curr = arr[idx]
        z_next = arr[min(K - 1, idx + 1)]
        slab = np.stack([z_prev, z_curr, z_next], axis=0).astype(np.float32) / 255.0
        slab = (slab - IMAGENET_MEAN) / IMAGENET_STD
        slabs.append(slab)
    return np.stack(slabs, axis=0).astype(np.float32)


class VolumeConsistentAugmenter:
    """Applies affine & photometric augmentations consistently across all slices in a plane.

    Task T5 (N2): When hflip=True, applies horizontal flip across all slices and planes of a study.
    Validated by Task T3 laterality audit: 54.6% Right vs 45.4% Left knees in the clinical dataset.
    Because Medial vs. Lateral is intrinsic joint anatomy (e.g. fibula is lateral, MCL is medial),
    reflecting a right knee horizontally produces a valid contralateral left knee with identical
    pathology-to-compartment mappings. Labels do not need to be swapped.
    """
    def __init__(self, p: float = 0.5, hflip: bool = False):
        self.p = p
        self.hflip = hflip

    def __call__(self, tensor: torch.Tensor, flip_this_volume: bool = False) -> torch.Tensor:
        """tensor shape: (K, 3, H, W) - K slices, 3 slab channels, H x W pixels."""
        if flip_this_volume:
            tensor = TF.hflip(tensor)

        if random.random() > self.p:
            return tensor

        angle = random.uniform(-7.0, 7.0)
        max_dx = int(0.05 * tensor.shape[-1])
        max_dy = int(0.05 * tensor.shape[-2])
        translations = [random.randint(-max_dx, max_dx), random.randint(-max_dy, max_dy)]
        scale = random.uniform(0.95, 1.05)

        contrast_factor = random.uniform(0.90, 1.10)
        brightness_factor = random.uniform(-0.08, 0.08)

        # Single vectorized affine call across all K slices in plane
        out = TF.affine(tensor, angle=angle, translate=translations, scale=scale, shear=[0.0, 0.0])
        out = out * contrast_factor + brightness_factor
        return out


class KneeMRITrainingDataset(Dataset):
    def __init__(self, labels_df: pd.DataFrame, series_df: pd.DataFrame, series_dir: str, num_slices: int = 16, augment: bool = False,
                 hflip: bool = False, report_vecs: Optional[Dict[str, np.ndarray]] = None, report_dim: int = 64):
        self.labels_df = labels_df.reset_index(drop=True)
        self.series_df = series_df
        self.series_dir = series_dir
        self.num_slices = num_slices
        self.augment = augment
        self.hflip = hflip
        self.report_vecs = report_vecs or {}
        self.report_dim = report_dim
        self.augmenter = VolumeConsistentAugmenter(p=0.5, hflip=hflip) if augment else None
        self.plane_col = None
        if not series_df.empty:
            for col in ["Anatomical_Plane", "anatomical_plane", "Plane", "SeriesDescription"]:
                if col in series_df.columns:
                    self.plane_col = col
                    break

        # Check for pre-cached .npz dataset
        self.cached_dir = None
        for cand in [
            "/kaggle/input/creaky-caching-dataset/cached_slices",
            "/kaggle/input/creaky-caching-dataset",
            "/kaggle/input/creaky-cached-slices",
            "/kaggle/input/cached-slices",
            "/kaggle/working/cached_slices",
            "data/processed/cached_slices"
        ]:
            if os.path.isdir(cand):
                self.cached_dir = cand
                print(f"--> [Fast Mode] Using pre-cached slices from: {cand}")
                break

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx: int):
        row = self.labels_df.iloc[idx]
        study_uid = str(row["StudyInstanceUID"])

        study_series = self.series_df[self.series_df["StudyInstanceUID"] == study_uid] if not self.series_df.empty else pd.DataFrame()

        # 1. Fast Path: Check for pre-cached .npz arrays
        cached_p = None
        if self.cached_dir:
            check_p = os.path.join(self.cached_dir, f"{study_uid}.npz")
            if os.path.exists(check_p):
                cached_p = check_p

        plane_tensors = {}
        if cached_p:
            try:
                npz = np.load(cached_p)
                for plane in ["Sagittal", "Coronal", "Axial"]:
                    arr = npz[plane.lower()]  # (K, H, W) uint8
                    plane_tensors[plane] = torch.tensor(cached_plane_to_slabs(arr), dtype=torch.float32)
            except Exception:
                plane_tensors = {}

        # 2. Raw DICOM Fallback
        if not plane_tensors:
            for plane in ["Sagittal", "Coronal", "Axial"]:
                series_files = []
                if not study_series.empty and self.plane_col is not None:
                    match = study_series[study_series[self.plane_col].astype(str).str.lower().str.contains(plane.lower())]
                    if not match.empty:
                        series_uid = str(match.iloc[0]["SeriesInstanceUID"])
                        pattern = os.path.join(self.series_dir, study_uid, series_uid, "*.dcm")
                        series_files = glob.glob(pattern)
                elif os.path.isdir(self.series_dir):
                    s_dir = os.path.join(self.series_dir, study_uid)
                    if os.path.isdir(s_dir):
                        subdirs = [d for d in os.listdir(s_dir) if os.path.isdir(os.path.join(s_dir, d))]
                        plane_idx = ["Sagittal", "Coronal", "Axial"].index(plane)
                        if plane_idx < len(subdirs):
                            series_files = glob.glob(os.path.join(s_dir, subdirs[plane_idx], "*.dcm"))

                plane_tensors[plane] = load_and_preprocess_series(series_files, target_size=(256, 256), num_slices=self.num_slices)

        # Apply Volume-Consistent Augmentation (train only, across all planes)
        if self.augmenter:
            flip_study = (self.hflip and random.random() < 0.5)
            for plane in ["Sagittal", "Coronal", "Axial"]:
                plane_tensors[plane] = self.augmenter(plane_tensors[plane], flip_this_volume=flip_study)

        targets = np.array([float(row[t]) for t in TARGET_COLS], dtype=np.float32)
        weights = []
        for t in TARGET_COLS:
            w_col = f"{t}_weight"
            w = float(row[w_col]) if (w_col in row and not pd.isna(row[w_col])) else (1.0 if row.get("label_source") == "gold" else 0.8)
            weights.append(w)
        weights = np.array(weights, dtype=np.float32)

        rvec = self.report_vecs.get(study_uid)
        has_report = rvec is not None
        if not has_report:
            rvec = np.zeros(self.report_dim, dtype=np.float32)

        return {
            "study_uid": study_uid,
            "sagittal": plane_tensors["Sagittal"],
            "coronal": plane_tensors["Coronal"],
            "axial": plane_tensors["Axial"],
            "targets": torch.tensor(targets, dtype=torch.float32),
            "weights": torch.tensor(weights, dtype=torch.float32),
            "report_vec": torch.tensor(rvec, dtype=torch.float32),
            "has_report": torch.tensor(float(has_report), dtype=torch.float32),
        }



# ==============================================================================
# 6. MODEL ARCHITECTURE (MATCHES INFERENCE 100%)
# ==============================================================================

class GatedAttentionPool(nn.Module):
    def __init__(self, in_features: int, hidden_dim: int = 128):
        super().__init__()
        self.v_proj = nn.Linear(in_features, hidden_dim)
        self.u_proj = nn.Linear(in_features, hidden_dim)
        self.w_proj = nn.Linear(hidden_dim, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        v = torch.tanh(self.v_proj(x))
        u = torch.sigmoid(self.u_proj(x))
        attn_scores = self.w_proj(v * u)
        attn_weights = F.softmax(attn_scores, dim=1)
        pooled = torch.sum(x * attn_weights, dim=1)
        return pooled

class KneeAnatomicalMoEClassifier(nn.Module):
    """Multi-Planar 2.5D Knee Model with Anatomical Plane-Specific Expert Routing.
    
    Routes anatomical representations according to clinical MR physics:
    - Sagittal plane -> ACL, Medial Meniscus, Lateral Meniscus heads
    - Coronal plane -> MCL head
    - Axial plane -> PF OA, Effusion, Synovitis heads
    - Combined multi-planar -> Medial OA, Lateral OA, Baker's, Contusion, Fracture heads
    """
    def __init__(
        self,
        backbone_name: str = "resnet34",
        pretrained: bool = True,
        num_classes: int = 12,
        dropout: float = 0.25
    ):
        super().__init__()
        self.num_classes = num_classes

        # Vision Backbone
        try:
            if backbone_name == "resnet34":
                weights = tv_models.ResNet34_Weights.DEFAULT if pretrained else None
                model = tv_models.resnet34(weights=weights)
                self.feat_dim = model.fc.in_features
                model.fc = nn.Identity()
                self.encoder = model
            elif backbone_name == "resnet50":
                weights = tv_models.ResNet50_Weights.DEFAULT if pretrained else None
                model = tv_models.resnet50(weights=weights)
                self.feat_dim = model.fc.in_features
                model.fc = nn.Identity()
                self.encoder = model
            elif backbone_name in ["convnext_tiny", "convnext_small"]:
                factory = tv_models.convnext_small if backbone_name == "convnext_small" else tv_models.convnext_tiny
                w_enum = tv_models.ConvNeXt_Small_Weights.DEFAULT if backbone_name == "convnext_small" else tv_models.ConvNeXt_Tiny_Weights.DEFAULT
                weights = w_enum if pretrained else None
                model = factory(weights=weights)
                self.feat_dim = model.classifier[2].in_features
                model.classifier[2] = nn.Identity()
                self.encoder = model
            elif backbone_name in ["efficientnet_v2_s", "effnet"]:
                weights = tv_models.EfficientNet_V2_S_Weights.DEFAULT if pretrained else None
                model = tv_models.efficientnet_v2_s(weights=weights)
                self.feat_dim = model.classifier[1].in_features
                model.classifier[1] = nn.Identity()
                self.encoder = model
            else:
                model = getattr(tv_models, backbone_name)(weights=None)
                self.feat_dim = model.fc.in_features
                model.fc = nn.Identity()
                self.encoder = model
        except Exception as e:
            print(f"Fallback to uninitialized backbone ({e})")
            model = tv_models.resnet34(weights=None)
            self.feat_dim = model.fc.in_features
            model.fc = nn.Identity()
            self.encoder = model

        self.sag_pool = GatedAttentionPool(self.feat_dim)
        self.cor_pool = GatedAttentionPool(self.feat_dim)
        self.ax_pool = GatedAttentionPool(self.feat_dim)

        combined_dim = self.feat_dim * 3

        self.sag_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim + combined_dim, 256),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 3)
        )

        self.cor_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim + combined_dim, 128),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

        self.ax_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.feat_dim + combined_dim, 256),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 3)
        )

        self.joint_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(combined_dim, 256),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 5)
        )

    def _encode_plane(self, x: torch.Tensor, pool_module: GatedAttentionPool) -> torch.Tensor:
        B, K, C, H, W = x.shape
        x_flat = x.view(B * K, C, H, W)
        feats = self.encoder(x_flat)
        if isinstance(feats, torch.Tensor) and feats.dim() > 2:
            feats = feats.flatten(1)
        feats = feats.view(B, K, self.feat_dim)
        return pool_module(feats)

    def forward(self, sag: torch.Tensor, cor: torch.Tensor, ax: torch.Tensor, return_features: bool = False):
        h_sag = self._encode_plane(sag, self.sag_pool)
        h_cor = self._encode_plane(cor, self.cor_pool)
        h_ax = self._encode_plane(ax, self.ax_pool)

        combined = torch.cat([h_sag, h_cor, h_ax], dim=1)

        out_sag = self.sag_head(torch.cat([h_sag, combined], dim=1))
        out_cor = self.cor_head(torch.cat([h_cor, combined], dim=1))
        out_ax = self.ax_head(torch.cat([h_ax, combined], dim=1))
        out_joint = self.joint_head(combined)

        logits = torch.stack([
            out_sag[:, 0],     # ACL
            out_cor[:, 0],     # MCL
            out_sag[:, 1],     # Medial Meniscus
            out_sag[:, 2],     # Lateral Meniscus
            out_joint[:, 0],   # Medial OA
            out_joint[:, 1],   # Lateral OA
            out_ax[:, 0],      # PF OA
            out_ax[:, 1],      # Effusion
            out_ax[:, 2],      # Synovitis
            out_joint[:, 2],   # Baker's
            out_joint[:, 3],   # Contusion
            out_joint[:, 4],   # Fracture
        ], dim=1)

        if return_features:
            return logits, combined
        return logits

# Backwards compatibility alias
KneeAbnormalityClassifier = KneeAnatomicalMoEClassifier



# ==============================================================================
# 7. LOSS & METRIC
# ==============================================================================

class WeightedBCEWithLogitsLoss(nn.Module):
    def __init__(self, label_smoothing: float = 0.05):
        super().__init__()
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor, weights: Optional[torch.Tensor] = None) -> torch.Tensor:
        if self.label_smoothing > 0:
            smoothed = targets * (1.0 - 2 * self.label_smoothing) + self.label_smoothing
        else:
            smoothed = targets
        bce = F.binary_cross_entropy_with_logits(logits, smoothed, reduction="none")
        if weights is not None:
            return (bce * weights).sum() / (weights.sum() + 1e-7)
        return bce.mean()

def compute_competition_metric(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, Dict[str, float]]:
    """Macro AUC over targets that have both classes present.

    Binarizes labels at 0.5 threshold (handles both discrete 0/1 gold and continuous silver probabilities),
    while excluding exact 0.5 unannotated soft-unknowns from evaluation. Targets with only one class
    are reported as NaN and left out of the mean.
    """
    per_class = {}
    valid_aucs = []
    for i, col in enumerate(TARGET_COLS):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        mask = (yt != 0.5)
        yt_masked = yt[mask]
        yp_masked = yp[mask]
        yt_bin = (yt_masked >= 0.5).astype(int)
        auc = float("nan")
        if len(np.unique(yt_bin)) == 2:
            try:
                auc = float(roc_auc_score(yt_bin, yp_masked))
                valid_aucs.append(auc)
            except ValueError as e:  # e.g. NaN predictions from a diverged model
                print(f"    [!] AUC failed for {col}: {e}")
        per_class[col] = auc
    macro = float(np.mean(valid_aucs)) if valid_aucs else float("nan")
    return macro, per_class


def label_counts(y_true: np.ndarray) -> Dict[str, Tuple[int, int]]:
    """(positives, negatives) per target, counting binary labels (>=0.5 positive, <0.5 negative, excluding 0.5)."""
    return {
        col: (int((y_true[:, i] > 0.5).sum()), int((y_true[:, i] < 0.5).sum()))
        for i, col in enumerate(TARGET_COLS)
    }


def print_split_report(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, Dict[str, float]]:
    """Print macro + per-target AUC for one validation subset, with label counts."""
    macro, per_class = compute_competition_metric(y_true, y_pred)
    counts = label_counts(y_true)
    n_scored = sum(not np.isnan(v) for v in per_class.values())
    print(f"  {name} AUC: {macro:.4f}  ({len(y_true)} studies, {n_scored}/{len(TARGET_COLS)} targets scorable)")
    for t in TARGET_COLS:
        pos, neg = counts[t]
        auc_txt = "   n/a" if np.isnan(per_class[t]) else f"{per_class[t]:.4f}"
        low = "  <- few positives, noisy" if 0 < pos < 5 else ""
        print(f"    {t:18s}: {auc_txt}  (pos {pos:4d} / neg {neg:4d}){low}")
    return macro, per_class


# ==============================================================================
# 8. MASTER TRAINING PIPELINE (PHASE 2: 5-FOLD MULTILABEL STRATIFIED CV)
# ==============================================================================

def assign_multilabel_folds(df: pd.DataFrame, target_cols: list, n_splits: int = 5, seed: int = 42) -> np.ndarray:
    """Zero-dependency iterative multilabel stratification (Seffke & Tsoumakas)."""
    np.random.seed(seed)
    n_samples = len(df)
    folds = np.full(n_samples, -1, dtype=int)

    Y = (df[target_cols].to_numpy() >= 0.5).astype(int)
    c_counts = Y.sum(axis=0)
    desired_fold_counts = np.zeros((n_splits, len(target_cols)))
    for c in range(len(target_cols)):
        desired_fold_counts[:, c] = c_counts[c] / n_splits

    desired_samples = n_samples / n_splits
    fold_samples = np.zeros(n_splits)
    current_fold_counts = np.zeros((n_splits, len(target_cols)))

    label_density = Y.sum(axis=1)
    sample_indices = np.argsort(-label_density)

    shuffled_indices = []
    for density in np.unique(label_density)[::-1]:
        group = sample_indices[label_density[sample_indices] == density]
        np.random.shuffle(group)
        shuffled_indices.extend(group)
    sample_indices = np.array(shuffled_indices)

    for idx in sample_indices:
        y = Y[idx]
        pos_labels = np.where(y == 1)[0]
        if len(pos_labels) > 0:
            rarest = min(pos_labels, key=lambda l: c_counts[l])
            shortfalls = desired_fold_counts[:, rarest] - current_fold_counts[:, rarest]
            best_fold = int(np.argmax(shortfalls))
        else:
            shortfalls = desired_samples - fold_samples
            best_fold = int(np.argmax(shortfalls))
        folds[idx] = best_fold
        fold_samples[best_fold] += 1
        current_fold_counts[best_fold] += y

    return folds


def build_report_fingerprints(train_csv: str, dim: int = 64, seed: int = 42) -> Dict[str, np.ndarray]:
    """E11: Turn each radiology report into a small L2-normalised vector ("fingerprint").

    Character n-gram TF-IDF works across the dataset's mix of languages (ES/NL/FR/TR/EL/BG/EN)
    without a tokenizer or internet. SVD squeezes it to `dim` numbers. Unsupervised, uses only
    report text (never labels), and only training-fold fingerprints are ever used as targets.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD

    df = pd.read_csv(train_csv)
    if "Report" not in df.columns:
        print("[report_aux] No 'Report' column in train.csv; disabling report supervision.")
        return {}
    df = df.dropna(subset=["Report"])
    texts = [strip_accents(str(t)).lower() for t in df["Report"]]
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=60000,
                            min_df=3, sublinear_tf=True, dtype=np.float32)
    X = tfidf.fit_transform(texts)
    svd = TruncatedSVD(n_components=dim, random_state=seed)
    Z = svd.fit_transform(X).astype(np.float32)
    Z /= (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-8)
    print(f"[report_aux] Built {len(Z)} report fingerprints (dim={dim}, "
          f"SVD explained variance={svd.explained_variance_ratio_.sum():.3f})")
    return {str(u): z for u, z in zip(df["StudyInstanceUID"].astype(str), Z)}


def run_training():
    train_csv, series_csv, series_dir = get_train_paths()
    if not train_csv or not os.path.exists(train_csv):
        raise FileNotFoundError("Could not locate train.csv in candidate paths.")

    # 1. Labels
    labels_df = assemble_labels(train_csv)
    series_df = pd.read_csv(series_csv) if (series_csv and os.path.exists(series_csv)) else pd.DataFrame()

    # 2. Phase 2: Multilabel Stratified Split
    # The 58 Gold studies serve as the fixed reference benchmark across all folds
    val_gold = labels_df[labels_df["is_gold"] == True].copy()
    silver_df = labels_df[labels_df["is_gold"] == False].reset_index(drop=True)

    n_splits = CONFIG.get("n_splits", 5)
    target_fold = CONFIG.get("fold", 0)

    silver_df["fold"] = assign_multilabel_folds(silver_df, TARGET_COLS, n_splits=n_splits, seed=42)

    train_df = silver_df[silver_df["fold"] != target_fold].reset_index(drop=True)
    val_silver = silver_df[silver_df["fold"] == target_fold].reset_index(drop=True)
    val_df = pd.concat([val_gold, val_silver], ignore_index=True)

    if CONFIG["max_train_studies"] is not None:
        train_df = train_df.iloc[:CONFIG["max_train_studies"]]
    if CONFIG.get("max_val_studies") is not None:
        val_df = val_df.iloc[:CONFIG["max_val_studies"]]

    print(f"\nPhase 2 Stratified Split — Fold {target_fold}/{n_splits}:")
    print(f"  Training Studies (Silver) : {len(train_df)}")
    print(f"  Validation Studies Total  : {len(val_df)} ({len(val_gold)} Gold + {len(val_silver)} Silver OOF)")
    print(f"  Augmentation Enabled      : {CONFIG.get('augment', True)}")

    # Row-aligned with val_loader output (shuffle=False), so predictions can be split by source.
    val_is_gold = val_df["is_gold"].fillna(False).astype(bool).to_numpy()
    gold_true = val_df.loc[val_is_gold, TARGET_COLS].to_numpy(dtype=np.float32)
    print("\nGold validation label counts (AUC on targets with <5 positives is mostly noise):")
    for t, (pos, neg) in label_counts(gold_true).items():
        print(f"    {t:18s}: pos {pos:3d} / neg {neg:3d}")

    # 3. DataLoaders
    report_dim = CONFIG.get("report_dim", 64)
    report_vecs = build_report_fingerprints(train_csv, dim=report_dim) if CONFIG.get("report_aux") else {}
    # Only training-fold studies get a report target; validation never sees report info.
    train_report_vecs = {u: report_vecs[u] for u in train_df["StudyInstanceUID"].astype(str) if u in report_vecs}
    use_report_aux = len(train_report_vecs) > 0
    print(f"[report_aux] enabled={use_report_aux}, train studies with report target: "
          f"{len(train_report_vecs)}/{len(train_df)}")

    train_ds = KneeMRITrainingDataset(
        train_df, series_df, series_dir, num_slices=CONFIG["num_slices"], augment=CONFIG.get("augment", True),
        hflip=CONFIG.get("hflip", False), report_vecs=train_report_vecs, report_dim=report_dim
    )
    val_ds = KneeMRITrainingDataset(
        val_df, series_df, series_dir, num_slices=CONFIG["num_slices"], augment=False,
        hflip=False, report_dim=report_dim
    )

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

    # 4. Model, Optimizer, Loss
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nTraining on Device: {device}")
    model = KneeAnatomicalMoEClassifier(
        backbone_name=CONFIG.get("backbone", "resnet34"),
        pretrained=CONFIG.get("pretrained", True),
        dropout=CONFIG["dropout"]
    ).to(device)
    # Report head lives OUTSIDE the model so the saved model_state_dict is identical in shape to
    # E04's and loads strictly in the unchanged submission notebook.
    report_head = nn.Sequential(
        nn.Dropout(CONFIG["dropout"]),
        nn.Linear(model.feat_dim * 3, 256),
        nn.SiLU(),
        nn.Linear(256, report_dim),
    ).to(device)
    aux_w = float(CONFIG.get("report_aux_weight", 0.5))
    params = list(model.parameters()) + (list(report_head.parameters()) if use_report_aux else [])
    optimizer = torch.optim.AdamW(params, lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-6)
    criterion = WeightedBCEWithLogitsLoss(label_smoothing=CONFIG["label_smoothing"])
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    select_metric = CONFIG.get("select_metric", "silver")
    if select_metric == "silver" and not (~val_is_gold).any():
        print("No silver validation studies; selecting checkpoints on gold AUC instead.")
        select_metric = "gold"
    best_val_auc = -1.0
    output_dir = "/kaggle/working" if os.path.exists("/kaggle/working") else "checkpoints"
    os.makedirs(output_dir, exist_ok=True)
    best_ckpt_path = os.path.join(output_dir, f"best_model_fold_{target_fold}.pt")
    val_preds_path = os.path.join(output_dir, f"val_preds_fold_{target_fold}.csv")
    accum_steps = CONFIG.get("accum_steps", 1)

    # 5. Epoch Loop
    print(f"\nStarting Training Loop (Fold {target_fold}/{n_splits})...")
    for epoch in range(1, CONFIG["epochs"] + 1):
        model.train()
        report_head.train()
        train_loss = 0.0
        train_cls_loss, train_aux_loss, aux_steps = 0.0, 0.0, 0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} [Train]")
        for step, batch in enumerate(pbar):
            sag = batch["sagittal"].to(device)
            cor = batch["coronal"].to(device)
            ax = batch["axial"].to(device)
            targets = batch["targets"].to(device)
            weights = batch["weights"].to(device)

            with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                if use_report_aux:
                    logits, feats = model(sag, cor, ax, return_features=True)
                else:
                    logits = model(sag, cor, ax)
                cls_loss = criterion(logits, targets, weights=weights)
                loss = cls_loss
            if use_report_aux:
                # Cosine distance to the report fingerprint, in fp32 for stability.
                mask = batch["has_report"].to(device)
                if mask.sum() > 0:
                    pred_vec = report_head(feats.float())
                    cos = F.cosine_similarity(pred_vec, batch["report_vec"].to(device), dim=1)
                    aux_loss = ((1.0 - cos) * mask).sum() / mask.sum()
                    loss = loss + aux_w * aux_loss
                    train_aux_loss += aux_loss.item()
                    aux_steps += 1
            train_cls_loss += cls_loss.item()
            loss = loss / accum_steps

            scaler.scale(loss).backward()

            if (step + 1) % accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            train_loss += loss.item() * accum_steps
            pbar.set_postfix({"loss": f"{loss.item() * accum_steps:.4f}"})

        scheduler.step()
        avg_train_loss = train_loss / max(1, len(train_loader))

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Validation
        model.eval()
        val_preds, val_targets, val_uids = [], [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{CONFIG['epochs']} [Val]"):
                sag = batch["sagittal"].to(device)
                cor = batch["coronal"].to(device)
                ax = batch["axial"].to(device)
                with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
                    logits = model(sag, cor, ax)
                    probs = torch.sigmoid(logits).cpu().numpy()
                val_preds.append(probs)
                val_targets.append(batch["targets"].cpu().numpy())
                val_uids.extend(batch["study_uid"])

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        y_true = np.vstack(val_targets)
        y_pred = np.vstack(val_preds)

        print(f"\n--- Epoch {epoch}/{CONFIG['epochs']} Summary (Fold {target_fold}) ---")
        print(f"  Train Loss : {avg_train_loss:.4f}")
        n_steps = max(1, len(train_loader))
        print(f"  Train cls loss: {train_cls_loss / n_steps:.4f}"
              + (f" | report aux loss (1-cos): {train_aux_loss / max(1, aux_steps):.4f}" if use_report_aux else ""))
        gold_auc, gold_per_class = print_split_report("Gold  ", y_true[val_is_gold], y_pred[val_is_gold])
        silver_auc, silver_per_class = float("nan"), {}
        if (~val_is_gold).any():
            silver_auc, silver_per_class = print_split_report("Silver", y_true[~val_is_gold], y_pred[~val_is_gold])
        primary_auc = silver_auc if select_metric == "silver" else gold_auc
        val_auc = primary_auc
        if np.isnan(val_auc):
            fallback_metric = "gold" if select_metric == "silver" else "silver"
            fallback_auc = gold_auc if select_metric == "silver" else silver_auc
            if not np.isnan(fallback_auc):
                print(f"  [Warning] {select_metric.capitalize()} AUC is NaN; falling back to {fallback_metric} AUC ({fallback_auc:.4f}) for selection.")
                val_auc = fallback_auc
        print(f"  Selection ({select_metric}) AUC: {val_auc:.4f} (Best so far: {max(best_val_auc, 0.0):.4f})")

        if not np.isnan(val_auc) and val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "architecture": "KneeAnatomicalMoEClassifier",
                "backbone": CONFIG.get("backbone", "resnet34"),
                "fold": target_fold,
                "n_splits": n_splits,
                "augment": CONFIG.get("augment", True),
                # Tells the submission notebook which preprocessing to reproduce at test time.
                "preprocessing": "cache_v1" if train_ds.cached_dir else "raw_v1",
                "val_auc": val_auc,
                "select_metric": select_metric,
                "gold_auc": gold_auc,
                "silver_auc": silver_auc,
                "gold_per_class_auc": gold_per_class,
                "silver_per_class_auc": silver_per_class,
                "config": CONFIG,
                "report_head_state_dict": report_head.state_dict() if use_report_aux else None,
            }, best_ckpt_path)
            # Validation predictions of the best epoch, for fitting ensemble weights later.
            preds_df = pd.DataFrame({"StudyInstanceUID": val_uids, "is_gold": val_is_gold})
            for i, t in enumerate(TARGET_COLS):
                preds_df[t] = y_true[:, i]
                preds_df[f"pred_{t}"] = y_pred[:, i]
            preds_df.to_csv(val_preds_path, index=False)
            print(f"  >>> Saved NEW BEST model to: {best_ckpt_path} ({select_metric} AUC: {best_val_auc:.4f})")
            print(f"  >>> Saved validation predictions to: {val_preds_path}")

    print(f"\nTraining Complete (Fold {target_fold})! Best {select_metric} validation macro AUC: {best_val_auc:.4f}")
    print(f"Saved Checkpoint: {best_ckpt_path}")



if __name__ == "__main__":
    run_training()

